# Name: Grant Christenson
### Date: 8/16/2025

<style>
.jp-Notebook {
    padding: var(--jp-notebook-padding);
    margin-left: 160px;
    outline: none;
    overflow: auto;
    background: var(--jp-layout-color0);
}
</style>

<img src="https://cdn.nba.com/logos/nba/1610612760/primary/L/logo.svg" alt="logo" style="position: fixed; top: -40px; left: 5px; height: 250px;">

# Answers  

## Part 1      

**Question 1:** 26 4-in-6 stretches in OKC's draft schedule.   

**Question 2:** 25.1 4-in-6 stretches on average.  

**Question 3:**    

- Most 4-in-6 stretches on average: CHA (28.1)    
- Fewest 4-in-6 stretches on average: NYK (22.2)       

**Question 4:** This is a written question. Please leave your response in the document under Question 4.          

**Question 5:**   

- BKN Defensive eFG%: 54.3%   
- When opponent on a B2B: 53.5%    

## Part 2  

Please show your work in the document, you don't need anything here.     

## Part 3    

**Question 9:**    

- Most Helped by Schedule: BOS (26.9 wins)     
- Most Hurt by Schedule: DET (-30.0 wins)     
  


# Setup and Data    

In [1]:
import pandas as pd
schedule = pd.read_csv("Data Project/schedule.csv")
draft_schedule = pd.read_csv("Data Project/schedule_24_partial.csv")
locations = pd.read_csv("Data Project/locations.csv")
game_data = pd.read_csv("Data Project/team_game_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Data Project/schedule.csv'

## Part 1 -- Schedule Analysis               

In this section, you're going to work to answer questions using NBA scheduling data.   

### Question 1  

**QUESTION:** How many times are the Thunder scheduled to play 4 games in 6 nights in the provided 80-game draft of the 2024-25 season schedule? (Note: clarification, the stretches can overlap, the question is really “How many games are the 4th game played over the past 6 nights?”)     



In [ ]:
#Looking for only OKC games
okc=draft_schedule[draft_schedule['team']=='OKC'].copy()
okc.head()

#Changing gamedate string to datetime
okc['gamedate']=pd.to_datetime(okc['gamedate'])
okc.head()

# New column that shows games in previous 6 days
okc['last_6']= [
    okc[(okc['gamedate']>= date - pd.Timedelta(days=5)) &
        (okc['gamedate']<= date)].shape[0]
    for date in okc['gamedate']
]
okc.head()

#counting up how many 4 games in 6 happens
total_4_in_6 = (okc['last_6']>= 4).sum()
print(total_4_in_6)

#List of the opponents
games_4_in_6 = okc[okc['last_6'] >= 4][['gamedate', 'opponent', 'last_6']]
print("\nGames that were the 4th (or more) in 6 nights:")
print(games_4_in_6.to_string(index=False))

<strong><span style="color:red">ANSWER 1:</span></strong>   

26 4-in-6 stretches in OKC's draft schedule.   

### Question 2     

**QUESTION:** From 2014-15 to 2023-24, what is the average number of 4-in-6 stretches for a team in a season? Adjust each team/season to per-82 games before taking your final average.   (I am assuming this is asking the same question as Q1 of: How many games are the 4th game played over the past 6 nights? )
  


In [ ]:
results = []
for (team, season), games in schedule.groupby(['team', 'season']):
    games = games.sort_values('gamedate').copy()
    games['gamedate'] = pd.to_datetime(games['gamedate'])

    last_6 = [((games['gamedate'] >= date - pd.Timedelta(days=5)) &
               (games['gamedate'] <= date)).sum() for date in games['gamedate']]
    total_4_in_6 = sum(x >= 4 for x in last_6)

    # Adjust to per-82 games
    total_games = len(games)
    per_82 = total_4_in_6 * 82 / total_games if total_games > 0 else 0

    results.append({'team': team, 'season': season, 'per_82': per_82})

avg_4_in_6 = pd.DataFrame(results)['per_82'].mean()
print(f"Average 4-in-6 stretches per team-season: {avg_4_in_6:.2f}")


<strong><span style="color:red">ANSWER 2:</span></strong>  

25.1 4-in-6 stretches on average.  

### Question 3  

**QUESTION:** Which of the 30 NBA teams has had the highest average number of 4-in-6 stretches between 2014-15 and 2023-24? Which team has had the lowest average? Adjust each team/season to per-82 games.     


In [ ]:
results = []

for (team, season), games in schedule.groupby(['team', 'season']):
    games = games.sort_values('gamedate').copy()
    games['gamedate'] = pd.to_datetime(games['gamedate'])
    last_6 = [((games['gamedate'] >= date - pd.Timedelta(days=5)) &
               (games['gamedate'] <= date)).sum() for date in games['gamedate']]
    total_4_in_6 = sum(x >= 4 for x in last_6)
    per_82 = total_4_in_6 * 82 / len(games) if len(games) > 0 else 0
    results.append({'team': team, 'season': season, 'per_82': per_82})

team_averages = pd.DataFrame(results).groupby('team')['per_82'].mean()

highest_avg_team = team_averages.idxmax()
highest_avg_value = team_averages.max()
lowest_avg_team = team_averages.idxmin()
lowest_avg_value = team_averages.min()

std_avg = team_averages.std()
mean_avg = team_averages.mean()

print(f"Standard deviation: {std_avg:.2f}")

print(f"\nOutlier analysis:")
print(f"{highest_avg_team} is {(highest_avg_value - mean_avg) / std_avg:.2f} standard deviations above the mean")
print(f"{lowest_avg_team} is {(mean_avg - lowest_avg_value) / std_avg:.2f} standard deviations below the mean")

print(f"Team with the highest average 4-in-6 stretches: {highest_avg_team} ({highest_avg_value:.2f})")
print(f"Team with the lowest average 4-in-6 stretches: {lowest_avg_team} ({lowest_avg_value:.2f})")

<strong><span style="color:red">ANSWER 3:</span></strong>  

- Most 4-in-6 stretches on average: CHA (28.1)     
- Fewest 4-in-6 stretches on average: NYK (22.2)       

### Question 4  

**QUESTION:** Is the difference between most and least from Q3 surprising, or do you expect that size difference is likely to be the result of chance?    

<strong><span style="color:red">ANSWER 4:</span></strong>  At first glance, the nearly six-game difference between Charlotte and New York in average 4th game in six nights does appear surprising. Given Charlotte's limited playoff success over the past decade, one could speculate that more challenging scheduling circumstances have been a contributing factor.

From a statistical standpoint, however, the gap falls just within the bounds of normal variation. Both teams sit close to two standard deviations from the league mean. With 30 teams in the league, it is expected that some teams will fall toward the tails of the distribution. That said, being nearly two standard deviations away suggests this is more than simple random chance and could reflect structural scheduling factors such as geography, arena availability, and market size.

In summary, while the difference is not so extreme as to be considered an outright scheduling anomaly, it is large enough to warrant attention. Over the course of a decade, a persistent six-game-per-season disparity can meaningfully impact competitive balance and player fatigue, and it is worth monitoring as the league continues to refine scheduling practices.

### Question 5   

**QUESTION:** What was BKN's defensive eFG% in the 2023-24 season? What was their defensive eFG% that season in situations where their opponent was on the second night of back-to-back?  

Formula for eFG%: (FGM+0.5*3PM)/FGA


Defensive- What opponents shoot against you


In [ ]:
# Finding Def eFG%
def calc_def_efg(data):
    return (data['fgmade'].sum() + 0.5 * data['fg3made'].sum()) / data['fgattempted'].sum()

# Filter Brooklyn games for 2023-24
bkn_games = game_data[(game_data['season'] == 2023) & (game_data['def_team'] == "BKN")].copy()
bkn_games['gamedate'] = pd.to_datetime(bkn_games['gamedate'])

# Prepare schedule data and flag back-to-backs
schedule['gamedate'] = pd.to_datetime(schedule['gamedate'])
schedule = schedule.sort_values(['team', 'gamedate'])
schedule['b2b'] = (schedule['gamedate'] - schedule.groupby('team')['gamedate'].shift(1)).dt.days == 1

# Merge with opponent schedule info
merged = bkn_games.merge(
    schedule[['season', 'gamedate', 'team', 'b2b']],
    left_on=['season', 'gamedate', 'off_team'],
    right_on=['season', 'gamedate', 'team'],
    how='left'
)

# Calculate results
overall_def_efg = calc_def_efg(bkn_games)
b2b_def_efg = calc_def_efg(merged[merged['b2b'] == True])

# Results
print(f"Brooklyn defensive eFG% (2023-24 overall): {overall_def_efg:.3f}")
print(f"Brooklyn defensive eFG% (2023-24 vs B2B opponents): {b2b_def_efg:.3f}")

<strong><span style="color:red">ANSWER 5:</span></strong>  

- BKN Defensive eFG%: 54.3%   
- When opponent on a B2B: 53.5%    

## Part 2 -- Trends and Visualizations                   

This is an intentionally open ended section, and there are multiple approaches you could take to have a successful project. Feel free to be creative. However, for this section, please consider only the density of games and travel schedule, not the relative on-court strength of different teams.    

### Question 6   

**QUESTION:** Please identify at least 2 trends in scheduling over time. In other words, how are the more recent schedules different from the schedules of the past? Please include a visual (plot or styled table) highlighting or explaining each trend and include a brief written description of your findings.  


Importing Packages, and Schedule Data Prep

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic

sns.set(style="whitegrid")

schedule['gamedate'] = pd.to_datetime(schedule['gamedate'])
schedule = schedule.sort_values(['team', 'gamedate']).reset_index(drop=True)


Finding Homestands and Roadtrips

In [ ]:
def identify_trips(team_data):
    team_data = team_data.copy()
    team_data['trip_change'] = team_data['home'].ne(team_data['home'].shift())
    team_data['trip_id'] = team_data['trip_change'].cumsum()
    trips = team_data.groupby(['trip_id', 'home']).size().reset_index(name='length')
    trips['team'] = team_data['team'].iloc[0]
    trips['season'] = team_data['season'].iloc[0]
    return trips

trips_df = pd.concat(
    [identify_trips(g) for _, g in schedule.groupby(['team','season'])],
    ignore_index=True
)

homestands = trips_df[trips_df['home'] == 1]
road_trips = trips_df[trips_df['home'] == 0]

homestand_avg = homestands.groupby('season')['length'].mean()
roadtrip_avg = road_trips.groupby('season')['length'].mean()


Finding Travel Distance of Each Team per Season

In [ ]:
schedule_loc = schedule.merge(
    locations[['team','latitude','longitude']], on='team'
).merge(
    locations[['team','latitude','longitude']], left_on='opponent', right_on='team',
    suffixes=('_team','_opp')
)

def calculate_team_travel_distance(team_data):
    team_data = team_data.sort_values('gamedate').reset_index(drop=True)
    total_distance = 0
    for i in range(1, len(team_data)):
        prev = team_data.iloc[i-1]
        curr = team_data.iloc[i]
        prev_loc = (prev['latitude_team'], prev['longitude_team']) if prev['home'] else (prev['latitude_opp'], prev['longitude_opp'])
        curr_loc = (curr['latitude_team'], curr['longitude_team']) if curr['home'] else (curr['latitude_opp'], curr['longitude_opp'])
        if prev_loc != curr_loc:
            total_distance += geodesic(prev_loc, curr_loc).miles
    return total_distance

travel_df = schedule_loc.groupby(['team_team','season']).apply(calculate_team_travel_distance).reset_index(name='total_distance')
travel_df.rename(columns={'team_team': 'team'}, inplace=True)


Road Trip Distance

In [ ]:
away_games = schedule_loc[schedule_loc['home']==0].sort_values(['team_team','season','gamedate'])

def road_trip_distances(df):
    df = df.reset_index(drop=True)
    distances = []
    for i in range(1,len(df)):
        prev_loc = (df.loc[i-1,'latitude_opp'], df.loc[i-1,'longitude_opp'])
        curr_loc = (df.loc[i,'latitude_opp'], df.loc[i,'longitude_opp'])
        distances.append(geodesic(prev_loc, curr_loc).miles)
    return pd.Series(distances)

road_trip_df = away_games.groupby(['team_team','season']).apply(road_trip_distances).reset_index()
road_trip_df.rename(columns={'level_2':'game_idx', 0:'distance'}, inplace=True)
avg_road_distance = road_trip_df.groupby('season')['distance'].mean()


Visuals

In [ ]:
fig, axes = plt.subplots(3,2,figsize=(16,18))
fig.suptitle('NBA Homestands, Road Trips & Travel Analysis', fontsize=16, fontweight='bold')

# Avg homestand vs road trip length
homestand_avg.plot(ax=axes[0,0], marker='o', color='blue', label='Homestands', linewidth=2)
roadtrip_avg.plot(ax=axes[0,0], marker='s', color='red', label='Road Trips', linewidth=2)
axes[0,0].set_title('Avg Homestand vs Road Trip Length')
axes[0,0].set_ylabel('Games')
axes[0,0].legend()

# Avg total travel distance per team per season
travel_df.groupby('season')['total_distance'].mean().plot(ax=axes[0,1], marker='o', color='green', linewidth=2)
axes[0,1].set_title('Avg Travel Distance per Team by Season')
axes[0,1].set_ylabel('Miles')

# Homestand length distribution
sns.histplot(homestands['length'], bins=range(1, homestands['length'].max()+2), color='blue', ax=axes[1,0])
axes[1,0].set_title('Homestand Length Distribution')

# Road trip length distribution
sns.histplot(road_trips['length'], bins=range(1, road_trips['length'].max()+2), color='red', ax=axes[1,1])
axes[1,1].set_title('Road Trip Length Distribution')

# Teams with most travel distance (recent season)
recent_travel = travel_df[travel_df['season']==travel_df['season'].max()].nlargest(10,'total_distance')
recent_travel.plot.barh(x='team', y='total_distance', ax=axes[2,0], color='orange')
axes[2,0].set_title(f'Most Travel - {travel_df["season"].max()} Season')

# Avg distance between consecutive away games
avg_road_distance.plot(ax=axes[2,1], marker='o', color='purple', linewidth=2)
axes[2,1].set_title('Avg Distance Between Consecutive Away Games')
axes[2,1].set_ylabel('Miles')

plt.tight_layout()
plt.show()


Trend 2: Holiday Games and Rest

Creating Holiday Game Indicator


In [ ]:
schedule['gamedate'] = pd.to_datetime(schedule['gamedate'])
schedule['month'] = schedule['gamedate'].dt.month
schedule['day'] = schedule['gamedate'].dt.day
schedule['year'] = schedule['gamedate'].dt.year

def is_holiday(row):
    month, day, year = row['month'], row['day'], row['year']
    date = pd.Timestamp(year, month, day)

    # Christmas
    if month == 12 and day == 25:
        return 'Christmas'

    # New Year's Day
    if month == 1 and day == 1:
        return 'New Year'

    # Thanksgiving (4th Thursday in November)
    if month == 11:
        first_of_month = pd.Timestamp(year, 11, 1)
        days_to_first_thursday = (3 - first_of_month.weekday()) % 7
        first_thursday = first_of_month + pd.Timedelta(days=days_to_first_thursday)
        fourth_thursday = first_thursday + pd.Timedelta(days=21)
        if date == fourth_thursday:
            return 'Thanksgiving'

    # MLK Day (3rd Monday in January)
    if month == 1:
        first_of_month = pd.Timestamp(year, 1, 1)
        days_to_first_monday = (0 - first_of_month.weekday()) % 7
        first_monday = first_of_month + pd.Timedelta(days=days_to_first_monday)
        third_monday = first_monday + pd.Timedelta(days=14)
        if date == third_monday:
            return 'MLK Day'

    return 'Regular'

schedule['holiday'] = schedule.apply(is_holiday, axis=1)
schedule['is_holiday'] = schedule['holiday'] != 'Regular'


Holiday Game Frequency

In [ ]:
holiday_counts = schedule[schedule['is_holiday']].groupby('team')['holiday'].count().sort_values(ascending=False)
holiday_breakdown = schedule[schedule['is_holiday']].groupby(['team','holiday']).size().unstack(fill_value=0)

# Example: top teams for holiday games
print("Top 10 Teams with Most Holiday Games:")
print(holiday_counts.head(10))


Holiday Rest

In [ ]:
# Calculate days of rest before each game
schedule = schedule.sort_values(['team','gamedate']).reset_index(drop=True)
schedule['days_rest'] = schedule.groupby('team')['gamedate'].diff().dt.days - 1

# Focus on holiday games (2014–2023)
holiday_rest = schedule[(schedule['is_holiday']) & (schedule['season'].between(2014, 2023))]

# Yearly average/median rest
yearly_avg_rest = holiday_rest.groupby('season')['days_rest'].agg(['mean','median']).round(2)

# By holiday type
holiday_type_rest = holiday_rest.groupby(['season','holiday'])['days_rest'].mean().unstack()


Rest Trends for Holiday Games

In [ ]:
import matplotlib.pyplot as plt

schedule['gamedate'] = pd.to_datetime(schedule['gamedate'])
schedule = schedule.sort_values(['team', 'gamedate']).reset_index(drop=True)
schedule['days_rest'] = schedule.groupby('team')['gamedate'].diff().dt.days - 1

# Focus on holiday games from 2014–2023 per rubric
holiday_rest = schedule[
    (schedule['is_holiday']) & (schedule['season'].between(2014, 2023))
].copy()

# Aggregations for the visuals and printed analysis
yearly_avg_rest = (
    holiday_rest.groupby('season')['days_rest']
    .agg(['mean', 'median'])
    .round(2)
)

holiday_type_rest = (
    holiday_rest.groupby(['season', 'holiday'])['days_rest']
    .mean()
    .unstack()
    .round(2)
)

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Rest Days Before Holiday Games (2014–2023)', fontsize=16, fontweight='bold')

# 1) Average + median rest over time
yearly_avg_rest.plot(ax=axes[0], marker='o', linewidth=2)
axes[0].set_title('Average Rest Days Before Holiday Games')
axes[0].set_ylabel('Days of Rest')
axes[0].grid(alpha=0.3)

# 2) Rest by holiday type over time
holiday_type_rest.plot(ax=axes[1], marker='o')
axes[1].set_title('Rest Days by Holiday Type')
axes[1].set_ylabel('Average Days of Rest')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=== HOLIDAY REST ANALYSIS (2014–2023) ===\n")

print("Average Rest Days by Season (mean / median):")
print(yearly_avg_rest)

if 2014 in yearly_avg_rest.index and 2023 in yearly_avg_rest.index:
    change = yearly_avg_rest.loc[2023, 'mean'] - yearly_avg_rest.loc[2014, 'mean']
    print(f"\nChange in Average Rest (2014 → 2023): {change:+.1f} days")
else:
    print("\nChange in Average Rest (2014 → 2023): N/A (endpoint season missing)")

# Simple trend signal
corr = yearly_avg_rest.index.to_series().corr(yearly_avg_rest['mean'])
print(f"Correlation between year and average rest: {corr:.3f}")

# Holiday type snapshot for the most recent year in range
latest_year = yearly_avg_rest.index.max()
recent_holiday_rest = (
    holiday_rest[holiday_rest['season'] == latest_year]
    .groupby('holiday')['days_rest']
    .agg(['mean', 'count'])
    .round(2)
)
print(f"\nRest by Holiday Type ({latest_year}):")
print(recent_holiday_rest)



Frequency and Distribution of Holiday Games

In [ ]:
schedule['gamedate'] = pd.to_datetime(schedule['gamedate'])
schedule['month'] = schedule['gamedate'].dt.month
schedule['day'] = schedule['gamedate'].dt.day
schedule['year'] = schedule['gamedate'].dt.year

def is_holiday(row):
    month, day, year = row['month'], row['day'], row['year']
    date = pd.Timestamp(year, month, day)

    # Christmas
    if month == 12 and day == 25:
        return 'Christmas'

    # New Year's Day
    if month == 1 and day == 1:
        return 'New Year'

    # Thanksgiving (4th Thursday in November)
    if month == 11:
        first_of_month = pd.Timestamp(year, 11, 1)
        days_to_first_thursday = (3 - first_of_month.weekday()) % 7
        first_thursday = first_of_month + pd.Timedelta(days=days_to_first_thursday)
        fourth_thursday = first_thursday + pd.Timedelta(days=21)
        if date == fourth_thursday:
            return 'Thanksgiving'

    # MLK Day (3rd Monday in January)
    if month == 1:
        first_of_month = pd.Timestamp(year, 1, 1)
        days_to_first_monday = (0 - first_of_month.weekday()) % 7
        first_monday = first_of_month + pd.Timedelta(days=days_to_first_monday)
        third_monday = first_monday + pd.Timedelta(days=14)
        if date == third_monday:
            return 'MLK Day'

    return 'Regular'

schedule['holiday'] = schedule.apply(is_holiday, axis=1)
schedule['is_holiday'] = schedule['holiday'] != 'Regular'


# Counts by team (all seasons in schedule)
holiday_counts = (
    schedule.loc[schedule['is_holiday']]
    .groupby('team')['holiday']
    .count()
    .sort_values(ascending=False)
)

# Breakdown by holiday type
holiday_breakdown = (
    schedule.loc[schedule['is_holiday']]
    .groupby(['team', 'holiday'])
    .size()
    .unstack(fill_value=0)
)

# Visuals: team holiday exposure & composition
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('NBA Teams Holiday Games Analysis', fontsize=16, fontweight='bold')

# (1) Total Holiday Games by Team (Top 15)
ax1 = axes[0, 0]
holiday_counts.head(15).plot(kind='bar', ax=ax1, color='darkred')
ax1.set_title('Teams with Most Holiday Games (All Years in Dataset)')
ax1.set_xlabel('Team')
ax1.set_ylabel('Number of Holiday Games')
ax1.tick_params(axis='x', rotation=45)

# (2) Holiday Games by Type (Top 15 teams by count)
ax2 = axes[0, 1]
holiday_breakdown.loc[holiday_counts.index[:15]].plot(
    kind='bar', stacked=True, ax=ax2, color=['gold', 'red', 'green', 'blue']
)
ax2.set_title('Holiday Games Breakdown by Team (Top 15)')
ax2.set_xlabel('Team')
ax2.set_ylabel('Number of Games')
ax2.legend(title='Holiday Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.tick_params(axis='x', rotation=45)

# (3) Christmas Games specifically
christmas_games = schedule.loc[schedule['holiday'] == 'Christmas', 'team'].value_counts()
ax3 = axes[1, 0]
christmas_games.head(10).plot(kind='barh', ax=ax3, color='red')
ax3.set_title('Christmas Day Games by Team')
ax3.set_xlabel('Number of Christmas Games')

fig.delaxes(axes[1, 1])

plt.tight_layout()
plt.show()

print("=== HOLIDAY GAMES ANALYSIS ===\n")

print("Top 10 Teams with Most Holiday Games:")
print(holiday_counts.head(10))

print(f"\nTotal holiday games across all teams: {schedule['is_holiday'].sum()}")
print(f"Average holiday games per team: {holiday_counts.mean():.1f}")

print("\nChristmas Games Leaders (Top 10):")
print(christmas_games.head(10))

# Holiday mix for Top 5 teams only
print("\n=== Top 5 Teams: Holiday Composition ===")
for team in holiday_counts.head(5).index:
    comp = holiday_breakdown.loc[team]
    comp = comp[comp > 0].sort_values(ascending=False)
    print(f"\n{team}:")
    for hol, cnt in comp.items():
        print(f"  {hol}: {cnt} games")

<strong><span style="color:red">ANSWER 6:</span></strong> See below






**Overview of Trend 1:**
This trend examines how the length of homestands, road trips, and overall travel distance has changed between the 2014 and 2023 seasons. The 2019–2020 season should be interpreted cautiously due to the COVID-19 pandemic and the Orlando bubble, which artificially reduced travel. This is visible in the trend graph titled “Avg Travel Distance per Team by Season.”

The analysis shows that average homestand and road trip lengths have increased slightly since 2020, aligning with an overall reduction in travel demands. In addition, the average distance between consecutive away games has declined significantly in recent years. These findings suggest that since 2020, the league has taken steps to improve scheduling efficiency, reducing both travel distance and fatigue associated with extended trips.


**Overview of Trend 2:** In this trend, I analyzed holiday games and how they affect rest between contests, focusing on Christmas, New Year’s Day, and Martin Luther King Jr. Day. The data confirms what we’d expect, teams like the Warriors, Lakers, and Knicks consistently appear in these marquee games. More importantly, the analysis shows that teams typically receive slightly more rest before holiday matchups, with the league average sitting just above one day of rest. Since 2014, that average has increased modestly by about 0.1 days, suggesting the NBA has made incremental adjustments to ensure players are better rested heading into high-profile, nationally televised games.


### Question 7    

**QUESTION:** Please design a plotting tool to help visualize a team’s schedule for a season. The plot should cover the whole season and should help the viewer contextualize and understand a team’s schedule, potentially highlighting periods of excessive travel, dense blocks of games, or other schedule anomalies. If you can, making the plots interactive (for example through the plotly package) is a bonus.   

Please use this tool to plot OKC and DEN's provided 80-game 2024-25 schedules.   

Installing Dash

In [ ]:
!pip install dash


Packages

In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import plotly.graph_objects as go
from datetime import timedelta

try:
    from dash import Dash, dcc, html
    DASH_AVAILABLE = True
except Exception:
    DASH_AVAILABLE = False


Data Wrangling, Adding Variables

In [ ]:
def prepare_team_schedule(schedule_df, locations_df, team_abbr, season):
    """
    Filter schedule to team/season and compute:
      - days_rest (between games)
      - travel_distance (miles between prior game site and current game site)
      - flags: back_to_back (0 rest), long_rest (>=3), 4-in-6 window count
      - cumulative_travel
    Returns processed team schedule (DataFrame).
    """
    df = schedule_df.copy()
    df['gamedate'] = pd.to_datetime(df['gamedate'])
    team = df[(df['team'] == team_abbr) & (df['season'] == season)].copy()
    team = team.sort_values('gamedate').reset_index(drop=True)

    # Merge locations for team (home arena) and opponent arena
    team = team.merge(
        locations_df[['team','latitude','longitude','timezone']],
        on='team', how='left'
    ).merge(
        locations_df[['team','latitude','longitude','timezone']],
        left_on='opponent', right_on='team', how='left', suffixes=('_team','_opp')
    )

    # Compute rest days (days between games minus 1)
    team['days_rest'] = team['gamedate'].diff().dt.days.sub(1).fillna(0).astype(int)

    # Travel distance from previous game site -> current game site
    def game_site_latlon(row, home_flag_col='home'):
        if row[home_flag_col] == 1:
            return (row['latitude_team'], row['longitude_team'])
        else:
            return (row['latitude_opp'], row['longitude_opp'])

    travel = [0.0]
    for i in range(1, len(team)):
        prev_site = game_site_latlon(team.iloc[i-1])
        curr_site = game_site_latlon(team.iloc[i])
        if prev_site == curr_site or None in prev_site or None in curr_site:
            travel.append(0.0)
        else:
            travel.append(geodesic(prev_site, curr_site).miles)
    team['travel_distance'] = np.round(travel, 1)

    # Flags and helpers
    team['game_number'] = np.arange(1, len(team) + 1)
    team['month'] = team['gamedate'].dt.month
    team['day_of_week'] = team['gamedate'].dt.day_name()
    team['back_to_back'] = (team['days_rest'] == 0)
    team['long_rest'] = (team['days_rest'] >= 3)
    team['cumulative_travel'] = team['travel_distance'].cumsum()

    # 4-in-6 density: for each game, count how many team games occurred in last 6 days (including today)
    dates = team['gamedate'].to_list()
    games_last_6 = []
    for i, d in enumerate(dates):
        window_start = d - timedelta(days=5)
        count = (team['gamedate'].between(window_start, d)).sum()
        games_last_6.append(count)
    team['games_last_6_days'] = games_last_6
    team['fourth_or_more_in_6'] = team['games_last_6_days'] >= 4

    return team


Season Plots

In [ ]:
def build_season_figures(team_df):
    # 1) Season timeline (rest & home/away)
    colors = ['#1f77b4' if h==1 else '#d62728' for h in team_df['home']]
    sizes = [10 + max(r, 0)*3 for r in team_df['days_rest']]

    timeline_fig = go.Figure()
    timeline_fig.add_trace(go.Scatter(
        x=team_df['gamedate'],
        y=[1]*len(team_df),
        mode='markers',
        marker=dict(size=sizes, color=colors, opacity=0.75, line=dict(width=1, color='black')),
        text=[
            f"Game {g}<br>vs {opp}<br>{'Home' if h==1 else 'Away'}"
            f"<br>Rest: {r} days<br>Travel: {t:.0f} mi"
            f"<br>4th+ in 6? {'Yes' if f else 'No'}"
            for g, opp, h, r, t, f in zip(
                team_df['game_number'], team_df['opponent'], team_df['home'],
                team_df['days_rest'], team_df['travel_distance'], team_df['fourth_or_more_in_6']
            )
        ],
        hovertemplate='%{text}<extra></extra>',
        name='Games'
    ))
    timeline_fig.update_layout(
        title="Season Timeline — Games & Rest",
        xaxis_title="Date",
        yaxis=dict(visible=False),
        height=300,
        margin=dict(l=30,r=30,t=60,b=30)
    )

    # 2) Travel per game
    travel_fig = go.Figure()
    travel_fig.add_trace(go.Scatter(
        x=team_df['game_number'], y=team_df['travel_distance'],
        mode='lines+markers',
        marker=dict(size=7),
        name='Travel per Game'
    ))
    travel_fig.update_layout(
        title="Travel Distance per Game",
        xaxis_title="Game Number", yaxis_title="Miles",
        height=300
    )

    # 3) Games by day of week
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    dow_counts = team_df['day_of_week'].value_counts()
    dow_counts = [dow_counts.get(d, 0) for d in day_order]

    dow_fig = go.Figure(go.Bar(x=day_order, y=dow_counts))
    dow_fig.update_layout(
        title="Games by Day of Week",
        xaxis_title="Day of Week", yaxis_title="Games",
        height=300
    )

    # 4) Monthly density
    month_names = ['', 'Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    monthly_counts = team_df['month'].value_counts().sort_index()
    month_fig = go.Figure(go.Bar(
        x=[month_names[m] for m in monthly_counts.index],
        y=monthly_counts.values
    ))
    month_fig.update_layout(
        title="Monthly Game Density",
        xaxis_title="Month", yaxis_title="Games",
        height=300
    )

    # 5) Difficulty map: rest vs travel (color by difficulty)
    def difficulty_color(r, t):
        if r <= 1 and t > 1000:
            return '#d62728'  # high difficulty
        elif r <= 1 or t > 1000:
            return '#ff7f0e'  # moderate
        return '#2ca02c'      # lower

    diff_colors = [difficulty_color(r, t) for r, t in zip(team_df['days_rest'], team_df['travel_distance'])]
    diff_fig = go.Figure(go.Scatter(
        x=team_df['days_rest'],
        y=team_df['travel_distance'],
        mode='markers',
        marker=dict(size=10, color=diff_colors, opacity=0.8),
        text=[f"Game {g} vs {opp}" for g, opp in zip(team_df['game_number'], team_df['opponent'])],
        hovertemplate='Rest: %{x} days<br>Travel: %{y:.0f} mi<br>%{text}<extra></extra>'
    ))
    diff_fig.update_layout(
        title="Schedule Difficulty Map (Rest vs Travel)",
        xaxis_title="Days of Rest", yaxis_title="Travel (miles)",
        height=300
    )

    # 6) Cumulative travel
    cum_fig = go.Figure(go.Scatter(
        x=team_df['gamedate'], y=team_df['cumulative_travel'],
        mode='lines',
        line=dict(width=3)
    ))
    cum_fig.update_layout(
        title="Cumulative Travel Over Season",
        xaxis_title="Date", yaxis_title="Cumulative Miles",
        height=300
    )

    return {
        'timeline_fig': timeline_fig,
        'travel_fig': travel_fig,
        'dow_fig': dow_fig,
        'month_fig': month_fig,
        'diff_fig': diff_fig,
        'cum_fig': cum_fig
    }


Difficulty Heatmap

In [ ]:
def build_difficulty_timeline(team_df):
    daily = team_df[['gamedate','travel_distance','days_rest','back_to_back','home']].copy()
    daily['intensity'] = (
        (daily['travel_distance'] / 1000.0) +
        (1.0 / (daily['days_rest'] + 1).clip(lower=1)) * 3 +
        daily['back_to_back'].astype(int) * 2 +
        (1 - daily['home']) * 0.5
    )

    fig = go.Figure(go.Scatter(
        x=daily['gamedate'],
        y=[1]*len(daily),
        mode='markers',
        marker=dict(
            size=18,
            color=daily['intensity'],
            colorscale='RdYlBu_r',
            showscale=True,
            colorbar=dict(title="Difficulty")
        ),
        text=[
            f"Date: {d.date()}<br>Rest: {r} days<br>Travel: {t:.0f} mi<br>{'Home' if h==1 else 'Away'}"
            for d, r, t, h in zip(daily['gamedate'], daily['days_rest'], daily['travel_distance'], daily['home'])
        ],
        hovertemplate='%{text}<extra></extra>'
    ))
    fig.update_layout(
        title="Season Schedule Difficulty Timeline",
        xaxis_title="Date",
        yaxis=dict(visible=False),
        height=280
    )
    return fig


Dash App

In [ ]:
def run_dash_app(team_df, team_abbr, season):
    if not DASH_AVAILABLE:
        raise RuntimeError("Dash is not available. Install dash to run the app.")

    figs = build_season_figures(team_df)
    heatmap_fig = build_difficulty_timeline(team_df)

    app = Dash(__name__)
    app.layout = html.Div([
        html.H1(f"{team_abbr} {season}-{str(season+1)[-2:]} Season Dashboard", style={'textAlign':'center'}),

        html.Div([
            html.H3("Season Summary"),
            html.Ul([
                html.Li(f"Total games: {len(team_df)}"),
                html.Li(f"Home games: {int(team_df['home'].sum())}"),
                html.Li(f"Away games: {int((1 - team_df['home']).sum())}"),
                html.Li(f"Total travel: {team_df['travel_distance'].sum():,.0f} miles"),
                html.Li(f"Back-to-back games: {int(team_df['back_to_back'].sum())}"),
                html.Li(f"4th+ in 6 nights: {int(team_df['fourth_or_more_in_6'].sum())}"),
                html.Li(f"Games with 3+ days rest: {int(team_df['long_rest'].sum())}")
            ])
        ], style={'padding':'8px 24px'}),

        dcc.Graph(figure=figs['timeline_fig']),
        dcc.Graph(figure=figs['travel_fig']),
        dcc.Graph(figure=figs['dow_fig']),
        dcc.Graph(figure=figs['month_fig']),
        dcc.Graph(figure=figs['diff_fig']),
        dcc.Graph(figure=figs['cum_fig']),
        dcc.Graph(figure=heatmap_fig),
    ])


    app.run(debug=False, port=8050)


2024-2025 80 Game Use

In [ ]:
schedule_24 = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/schedule_24_partial.csv")
locations   = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/locations.csv")

# OKC
okc_2425 = prepare_team_schedule(schedule_24, locations, team_abbr='OKC', season=2024)
okc_figs  = build_season_figures(okc_2425)
okc_heat  = build_difficulty_timeline(okc_2425)

# Display OKC figures (inline, no Dash)
for key, fig in okc_figs.items():
    fig.show()
okc_heat.show()



Running App

In [ ]:
run_dash_app(okc_2425, team_abbr='OKC', season=2024)

2023 Analysis or any other season

In [ ]:
# Load your provided files (update paths as needed)
schedule  = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/schedule.csv")
locations = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/locations.csv")

# OKC 2023-24
okc_2324 = prepare_team_schedule(schedule, locations, team_abbr='OKC', season=2023)
okc_figs = build_season_figures(okc_2324)
okc_heat = build_difficulty_timeline(okc_2324)

# Display OKC figures (inline, no Dash)
for fig in okc_figs.values():
    fig.show()
okc_heat.show()

Run App

In [ ]:
run_dash_app(okc_2324, team_abbr='OKC', season=2023)

**Side Note:** Could be used for other teams and seasons as well

### Question 8    

**QUESTION:** Using your tool, what is the best and worst part of OKC’s 2024-25 draft schedule? Please give your answer as a short brief to members of the front office and coaching staff to set expectations going into the season. You can include context from past schedules.  



<strong><span style="color:red">ANSWER 8:</span></strong>  

The most favorable stretch falls right at the start of the season, Oct 24 – Nov 2. This opening run includes several home games and relatively light travel compared to other parts of the schedule. Rest is balanced, with multiple one- and two-day breaks built in. Importantly, cumulative travel has not yet ramped up, which provides an opportunity to establish rhythm before heavier travel loads kick in later. For a young team like OKC, starting with manageable travel is an advantage in building early momentum.

The toughest part of the schedule comes in the lead-up to the New Year ,Dec 20 – Jan 8. This block combines heavy travel  with short rest between games. The stretch also contains multiple back-to-backs and high-caliber Eastern Conference opponents. Fatigue risk is especially high: this period has several “4-in-6” scenarios, and it comes with minimal long rest opportunities.

The Thunder’s 2024–25 draft schedule is front-loaded with a relatively favorable October/November stretch but becomes grueling through the late December–early January window, when travel spikes and back-to-backs pile up. From a planning standpoint, the front office and coaching staff should anticipate using the strong start to build cushion in the standings, while carefully managing workloads through the December–January grind to keep the roster healthy for the second half.

## Part 3 -- Modeling     

### Question 9   

**QUESTION:** Please estimate how many more/fewer regular season wins each team has had due to schedule-related factors from 2019-20 though 2023-24. Your final answer should have one number for each team, representing the total number of wins (not per 82, and not a per-season average). You may consider the on-court strength of the scheduled opponents as well as the impact of travel/schedule density. Please include the teams and estimates for the most helped and most hurt in the answer key.    

If you fit a model to help answer this question, please write a paragraph explaining your model, and include a simple model diagnostic (eg a printed summary of a regression, a variable importance plot, etc).    

Installing Necessary Packages

In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

USE_WIN_PCT_TARGET = False
VERBOSE_PRINT_LIMIT = 10
RANDOM_STATE = 42


Data Compilation, Adding Variables

In [ ]:
def _normalize_home_flag(series: pd.Series) -> pd.Series:
    s = series.copy()
    if not np.issubdtype(s.dtype, np.number):
        if s.dtype == object:
            mapping = {"H": 1, "A": 0, "Home": 1, "Away": 0}
            s = s.map(mapping).fillna(s)
        s = pd.to_numeric(s, errors="coerce")
    return s.fillna(0).astype(int)

def _coord_missing(coord) -> bool:
    if coord is None or not isinstance(coord, (tuple, list)) or len(coord) != 2:
        return True
    lat, lon = coord
    return pd.isna(lat) or pd.isna(lon)

def _compute_team_season(grp: pd.DataFrame) -> pd.DataFrame:
    df = grp.sort_values("gamedate").reset_index(drop=True)

    # Rest features
    df["days_rest"]  = df["gamedate"].diff().dt.days - 1
    df["b2b"]        = (df["days_rest"] == 0).fillna(False).astype(int)
    df["short_rest"] = (df["days_rest"] <= 1).fillna(False).astype(int)
    df["long_rest"]  = (df["days_rest"] >= 3).fillna(False).astype(int)

    # Travel (previous venue -> current venue)
    def venue_latlon(row):
        if row["home"] == 1:
            return (row["latitude_team"], row["longitude_team"])
        else:
            return (row["latitude_opp"], row["longitude_opp"])

    travel = [0.0]
    for i in range(1, len(df)):
        c1 = venue_latlon(df.iloc[i-1])
        c2 = venue_latlon(df.iloc[i])
        if _coord_missing(c1) or _coord_missing(c2) or c1 == c2:
            miles = 0.0
        else:
            miles = geodesic(c1, c2).miles
        travel.append(miles)
    df["travel_distance"] = np.round(travel, 1)

    # 4-in-6 fatigue: how many games in the past 6 nights (incl. current)
    dates = df["gamedate"].to_numpy()
    games_last_6 = []
    for i, d in enumerate(dates):
        start = d - np.timedelta64(5, "D")
        count = int(((dates >= start) & (dates <= d)).sum())
        games_last_6.append(count)
    df["games_last_6"] = games_last_6
    df["fatigue_4in6"] = (df["games_last_6"] >= 4).astype(int)

    return df

def calculate_schedule_factors(schedule: pd.DataFrame, locations: pd.DataFrame,
                               season_min: int = 2019, season_max: int = 2023) -> pd.DataFrame:
    # Normalize columns & filter seasons
    sched = schedule.copy()
    locs  = locations.copy()
    sched.columns = sched.columns.str.strip().str.lower()
    locs.columns  = locs.columns.str.strip().str.lower()
    sched = sched[sched["season"].between(season_min, season_max)].copy()

    # Normalize home flag & dates
    sched["home"]     = _normalize_home_flag(sched["home"])
    sched["gamedate"] = pd.to_datetime(sched["gamedate"])

    # Merge coordinates
    loc_team = locs.rename(columns={"latitude": "latitude_team", "longitude": "longitude_team"})
    sched = sched.merge(loc_team[["team", "latitude_team", "longitude_team"]], on="team", how="left")

    loc_opp = locs.rename(columns={"team": "opponent", "latitude": "latitude_opp", "longitude": "longitude_opp"})
    sched = sched.merge(loc_opp[["opponent", "latitude_opp", "longitude_opp"]], on="opponent", how="left")

    # Per-team-season time series features
    processed = []
    for (tm, ssn), grp in sched.groupby(["team", "season"], sort=False):
        processed.append(_compute_team_season(grp))
    full = pd.concat(processed, ignore_index=True)

    # Opponent win% (required opponent strength control)
    team_records = (
        full.groupby(["team", "season"])
        .agg(wins=("win", "sum"), games=("win", "size"))
        .reset_index()
    )
    team_records["win_pct"] = team_records["wins"] / team_records["games"]
    opp_records = team_records[["team", "season", "win_pct"]].rename(
        columns={"team": "opponent", "win_pct": "win_pct_opp"}
    )
    full = full.merge(opp_records, on=["opponent", "season"], how="left")

    # Aggregate to team-season factors
    factors = (
        full.groupby(["team", "season"], as_index=False)
        .agg(
            avg_rest=("days_rest", "mean"),
            b2b_games=("b2b", "sum"),
            short_rest_games=("short_rest", "sum"),
            long_rest_games=("long_rest", "sum"),
            fatigue_games=("fatigue_4in6", "sum"),
            total_travel_miles=("travel_distance", "sum"),
            home_games=("home", "sum"),
            avg_opponent_win_pct=("win_pct_opp", "mean"),
            actual_wins=("win", "sum"),
            total_games=("win", "size"),
        )
    )

    # Diagnostics / derived
    factors["home_game_pct"]       = factors["home_games"] / factors["total_games"]
    factors["b2b_rate"]            = factors["b2b_games"] / factors["total_games"]
    factors["avg_travel_per_game"] = factors["total_travel_miles"] / factors["total_games"]
    factors["win_pct"]             = factors["actual_wins"] / factors["total_games"]

    return factors


Modeling

In [ ]:
def build_schedule_impact_models(schedule_factors: pd.DataFrame):
    # Keep core, non-redundant features
    features = [
        "avg_rest",
        "b2b_games",
        "fatigue_games",
        "total_travel_miles",
        "home_game_pct",
        "avg_opponent_win_pct",
    ]

    X = schedule_factors[features].fillna(0.0)
    y = schedule_factors["actual_wins"].astype(float) if not USE_WIN_PCT_TARGET else schedule_factors["win_pct"].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE)

    models = {
        "Linear Regression": LinearRegression(),
        # Constrained RF to avoid extreme counterfactual swings
        "Random Forest": RandomForestRegressor(
            n_estimators=400,
            max_depth=6,
            min_samples_leaf=3,
            random_state=RANDOM_STATE
        ),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results[name] = {
            "model": model,
            "r2":   r2_score(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        }

    best_name  = max(results, key=lambda k: results[k]["r2"])
    best_model = results[best_name]["model"]
    return best_model, features, results, best_name, (X_train, X_test, y_train, y_test)

def plot_variable_importance(best_model, best_name, feature_names, top_n=12):
    if best_name == "Random Forest" and hasattr(best_model, "feature_importances_"):
        imp = pd.DataFrame({"feature": feature_names, "importance": best_model.feature_importances_}) \
               .sort_values("importance", ascending=False).head(top_n)
        plt.figure(figsize=(7, max(3, 0.35*len(imp))))
        sns.barplot(data=imp, x="importance", y="feature")
        plt.title("Variable Importance — Random Forest")
        plt.tight_layout()
        plt.show()
    elif best_name == "Linear Regression" and hasattr(best_model, "coef_"):
        coef = pd.DataFrame({"feature": feature_names, "importance": np.abs(best_model.coef_)}) \
               .sort_values("importance", ascending=False).head(top_n)
        plt.figure(figsize=(7, max(3, 0.35*len(coef))))
        sns.barplot(data=coef, x="importance", y="feature")
        plt.title("Variable Importance — Linear Regression (|coef|)")
        plt.tight_layout()
        plt.show()


Neutral Schedule

In [ ]:
def calculate_schedule_adjusted_wins(schedule_factors: pd.DataFrame, model, features: list) -> pd.DataFrame:

    df = schedule_factors.copy()

    # Season-wise neutralization
    neutral = df.copy()
    for c in features:
        neutral[c] = df.groupby("season")[c].transform("mean")

    X_actual  = df[features].fillna(0.0)
    X_neutral = neutral[features].fillna(0.0)

    df["predicted_wins"] = model.predict(X_actual)
    df["neutral_wins"]   = model.predict(X_neutral)

    # Clip to valid range
    tg = df["total_games"].astype(float)
    df["predicted_wins"] = df["predicted_wins"].clip(lower=0, upper=tg)
    df["neutral_wins"]   = df["neutral_wins"].clip(lower=0, upper=tg)

    # Impact (wins per season)
    df["impact_wins_this_season"] = df["predicted_wins"] - df["neutral_wins"]

    # Plausibility cap
    df["impact_wins_this_season"] = df["impact_wins_this_season"].clip(-6, 6)

    return df

def sanity_report(adjusted_df: pd.DataFrame):
    g = adjusted_df["impact_wins_this_season"]
    print("\n--- Sanity Report (per-season impacts) ---")
    print(f"Min/Max: {g.min():.2f}/{g.max():.2f}   Mean±SD: {g.mean():.2f}±{g.std():.2f}")
    hits = ((np.isclose(g, 6.0)) | (np.isclose(g, -6.0))).sum()
    print("At cap (±6 wins):", int(hits))

Report

In [ ]:
def analyze_results(results_df: pd.DataFrame, top_k: int = VERBOSE_PRINT_LIMIT) -> pd.DataFrame:
    by_team = (
        results_df.groupby("team", as_index=False)
        .agg(
            seasons=("season", "nunique"),
            impact_sum_wins=("impact_wins_this_season", "sum"),
            impact_mean_wins=("impact_wins_this_season", "mean"),
        )
        .sort_values("impact_sum_wins", ascending=False)
        .reset_index(drop=True)
    )

    print("=== Schedule Impact (Wins) — Multi-year Summary ===")
    print("Most HELPED :",
          by_team.iloc[0]["team"],
          f"→ sum={by_team.iloc[0]['impact_sum_wins']:.1f} over {int(by_team.iloc[0]['seasons'])} seasons",
          f"(avg={by_team.iloc[0]['impact_mean_wins']:.2f}/season)")
    print("Most HURT   :",
          by_team.iloc[-1]["team"],
          f"→ sum={by_team.iloc[-1]['impact_sum_wins']:.1f} over {int(by_team.iloc[-1]['seasons'])} seasons",
          f"(avg={by_team.iloc[-1]['impact_mean_wins']:.2f}/season)")

    print(f"\nTop {top_k} by multi-year sum (wins):")
    print(by_team.head(top_k)[["team", "impact_sum_wins", "impact_mean_wins"]])

    print(f"\nBottom {top_k} by multi-year sum (wins):")
    print(by_team.tail(top_k)[["team", "impact_sum_wins", "impact_mean_wins"]])

    return by_team

Main Analysis

In [ ]:
def run_schedule_impact_analysis(schedule: pd.DataFrame, locations: pd.DataFrame,
                                 season_min: int = 2019, season_max: int = 2023):
    # 1) Features
    factors = calculate_schedule_factors(schedule, locations, season_min, season_max)

    # 2) Modeling
    best_model, features, results, best_name, (X_train, X_test, y_train, y_test) = \
        build_schedule_impact_models(factors)

    # 3) Counterfactual (wins only)
    adjusted = calculate_schedule_adjusted_wins(factors, best_model, features)

    # 4) Quick sanity check
    sanity_report(adjusted)

    # 5) Reporting
    totals = analyze_results(adjusted)

    # 6) Diagnostics
    print("\n=== Model Diagnostics ===")
    for name, res in results.items():
        print(f"{name}: R²={res['r2']:.3f}, RMSE={res['rmse']:.3f}")
    print(f"\nSelected best model: {best_name}")

    # 7) Variable importance
    plot_variable_importance(best_model, best_name, features, top_n=12)

    return totals, adjusted, (best_name, best_model)


Running

In [ ]:
schedule  = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/schedule.csv")
locations = pd.read_csv("/content/drive/MyDrive/Thunder Data Project/locations.csv")

totals, per_team_season, (best_name, best_model) = run_schedule_impact_analysis(
     schedule, locations, season_min=2019, season_max=2023
 )


<strong><span style="color:red">ANSWER 9:</span></strong>  

- Most Helped by Schedule: BOS (26.9 wins)    
- Most Hurt by Schedule: DET (-30.0 wins)    


Model Explaination:
In order to get insights into how scheduling may benefit or harm an NBA team, I built two models, a Linear Regression model and a Random Forest model for predictions. I pinpointed some of the main aspects that may benefit or harm an NBA team, such as rest time between games, back to back games, the previously mentioned 4 in 6 games(called fatigue games), travel distance, and opponent strength. I then selected the better of the two models with R squared and Root Mean Sqaured error. In the chart above we can see which variables mattered most. In order to create predictions, I ran predictions on a neutral/average schedule to see which teams added more wins or less wins due to scheduling.

Side note: I added some guardrails like a maximum of 6 games +/-. Hopefully this helps reduce some noise. It seems to struggle with the most hurt by scheduling.